In [ ]:
# Binder Cumulant — Erdős-Rényi Model (Beta Sweep)

Analyse the effect of **β** on the critical temperature of the Erdős-Rényi model
using the **Binder cumulant** (4th-order cumulant ratio):

$$
U_4(\beta, N) = 1 - \frac{\langle m^4 \rangle}{3\,\langle m^2 \rangle^2}
$$

where $m = M_t / N$ is the normalised magnetisation.

The critical temperature $\beta_c$ is identified at the **intersection point** of
$U_4$ curves for different system sizes $N$ (finite-size scaling).

**Simulation parameters**
| Parameter | Value |
|-----------|-------|
| α         | 20    |
| β range   | 0.20 … 0.30 (step 0.02) |
| N         | 5 000, 10 000, 50 000 |
| MC sweeps | 100 000 |

Run `research/experiments/binder_cumulant_beta_sweep.py` first to generate the data.

In [ ]:
import os
import glob
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib_inline.backend_inline import set_matplotlib_formats

set_matplotlib_formats("svg")
plt.style.use("default")
pd.set_option("mode.copy_on_write", True)

%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
# ── Locate the latest binder_cumulant_beta_sweep result directory ──────────
RESULTS_ROOT = os.path.join(
    os.path.dirname(os.path.abspath("__file__")),
    "../../results/erdos_renyi"
)

sweep_dirs = sorted(
    glob.glob(os.path.join(RESULTS_ROOT, "binder_cumulant_beta_sweep_*"))
)

if not sweep_dirs:
    raise FileNotFoundError(
        "No binder_cumulant_beta_sweep_* directory found inside "
        f"{RESULTS_ROOT}.\n"
        "Please run  research/experiments/binder_cumulant_beta_sweep.py  first."
    )

latest_dir = sweep_dirs[-1]
print(f"Loading from: {latest_dir}")

# ── Load metadata ────────────────────────────────────────────────────────────
with open(os.path.join(latest_dir, "metadata.json")) as f:
    meta = json.load(f)

print(json.dumps(meta["parameters"], indent=2))

# ── Load magnetisation time series ──────────────────────────────────────────
mag_path = os.path.join(latest_dir, "magnetization.csv")
magnetization = pd.read_csv(mag_path, header=[0, 1], index_col=0)
magnetization.columns = magnetization.columns.set_levels([
    magnetization.columns.levels[0].astype(int),    # N
    magnetization.columns.levels[1].astype(float),  # beta
])

print(f"\nLoaded magnetization.csv  —  shape: {magnetization.shape}")
print(f"N values : {magnetization.columns.get_level_values('N').unique().tolist()}")
print(f"β values : {magnetization.columns.get_level_values('beta').unique().tolist()}")

In [ ]:
def binder_cumulant(m_series: np.ndarray, burn_in: float = 0.2) -> float:
    """
    Compute the 4th-order Binder cumulant:

        U4 = 1 - <m^4> / (3 * <m^2>^2)

    Parameters
    ----------
    m_series : normalised magnetisation time series  (M_t / N)
    burn_in  : fraction of the series to discard as thermalisation
    """
    start = int(len(m_series) * burn_in)
    m = m_series[start:]
    m2 = np.mean(m ** 2)
    m4 = np.mean(m ** 4)
    if m2 == 0:
        return np.nan
    return float(1.0 - m4 / (3.0 * m2 ** 2))


# ── Compute U4 for every (N, beta) combination ──────────────────────────────
N_values    = sorted(magnetization.columns.get_level_values("N").unique())
beta_values = sorted(magnetization.columns.get_level_values("beta").unique())

records = []
for N in N_values:
    for beta in beta_values:
        raw_M_t = magnetization[(N, beta)].dropna().values
        m_t = raw_M_t / N          # normalise to [-1, 1]
        u4  = binder_cumulant(m_t)
        records.append({"N": N, "beta": beta, "U4": u4})

binder_df = pd.DataFrame(records)
binder_df

In [ ]:
# ── Plot U4(beta, N) for different N ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))

markers   = ["o", "s", "^", "D", "v", "P"]
colors    = plt.cm.tab10.colors

for idx, N in enumerate(N_values):
    subset = binder_df[binder_df["N"] == N].sort_values("beta")
    ax.plot(
        subset["beta"],
        subset["U4"],
        marker=markers[idx % len(markers)],
        color=colors[idx % len(colors)],
        linewidth=1.8,
        markersize=6,
        label=f"$N = {N:,}$",
    )

# ── Reference lines ──────────────────────────────────────────────────────────
ax.axhline(0,    color="grey", linewidth=0.8, linestyle="--", alpha=0.6)
ax.axhline(2/3,  color="grey", linewidth=0.8, linestyle=":",  alpha=0.6,
           label=r"$U_4 = 2/3$ (ordered)")

# ── Labels & formatting ──────────────────────────────────────────────────────
ax.set_xlabel(r"$\beta$", fontsize=14)
ax.set_ylabel(r"$U_4(\beta,\, N)$", fontsize=14)
ax.set_title(
    r"Binder Cumulant vs $\beta$ (Erdős-Rényi, $\alpha=20$)",
    fontsize=13,
)
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(latest_dir, "binder_cumulant.svg"), dpi=150)
plt.show()
print("Figure saved to:", os.path.join(latest_dir, "binder_cumulant.svg"))

In [ ]:
# ── Pivot table: U4 values for each (N, beta) ────────────────────────────────
pivot = binder_df.pivot(index="beta", columns="N", values="U4")
pivot.columns.name = "N"
pivot.index.name   = "β"
pivot.round(4)